# Phase 2: Real-World Dataset Preprocessing & Transformation Pipeline
## Statlog (German Credit Data) — Preprocessing & Leakage-Free Pipeline
### Credit Risk & Loan Decision Intelligence Platform

---

## SECTION 1 — Objective & Preprocessing Principles

Data preprocessing prepares raw tabular data for machine learning algorithms while enforcing strict statistical hygiene:
1. **Strict Data Leakage Prevention**: Transformers (`StandardScaler`, `OneHotEncoder`) must be fitted **only on the training partition** ($X_{train}$). Test data ($X_{test}$) is strictly transformed using the parameters learned from training.
2. **Dynamic Feature Identification**: Numerical and categorical columns are dynamically discovered from the dataset schema.
3. **Stratified Splitting**: An 80/20 train/test split preserving the exact class proportion of the target variable (`default`).
4. **Automated Validation Gates**: Programmatic assertions verifying processed matrix properties (no NaNs, no infinities, numeric dtype, target separation).

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(f"Numpy version : {np.__version__}")
print(f"Pandas version: {pd.__version__}")

Numpy version : 2.4.6
Pandas version: 3.0.3


---
## SECTION 2 — Load Raw Dataset & Separate Target

We load the raw CSV and separate the predictive feature matrix $X$ from the ground-truth target vector $y$.

In [2]:
data_path = '../data/raw/german_credit_1000.csv' if os.path.exists('../data/raw/german_credit_1000.csv') else 'data/raw/german_credit_1000.csv'
df = pd.read_csv(data_path)

X = df.drop(columns=['default'])
y = df['default']

print(f"Full Dataset Dimensions:")
print(f"  • Feature Matrix X shape : {X.shape}")
print(f"  • Target Vector y shape   : {y.shape}")

Full Dataset Dimensions:
  • Feature Matrix X shape : (1000, 20)
  • Target Vector y shape   : (1000,)


---
## SECTION 3 — Dynamic Feature Type Identification

We detect numerical and categorical feature sets dynamically from the pandas dtypes.

In [3]:
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Dynamically Detected Features:")
print(f"  • Numerical Features ({len(numerical_features)}):")
for f in numerical_features:
    print(f"      - {f}")
print(f"  • Categorical Features ({len(categorical_features)}):")
for f in categorical_features:
    print(f"      - {f}")

Dynamically Detected Features:
  • Numerical Features (7):
      - duration_in_months
      - credit_amount
      - installment_rate_pct_disposable_income
      - present_residence_since
      - age_years
      - existing_credits_count
      - people_liable_maintenance
  • Categorical Features (13):
      - status_existing_checking_account
      - credit_history
      - purpose
      - savings_account_bonds
      - present_employment_since
      - personal_status_sex
      - other_debtors_guarantors
      - property
      - other_installment_plans
      - housing
      - job
      - telephone
      - foreign_worker


---
## SECTION 4 — Stratified 80/20 Train/Test Partition

We perform an 80/20 train/test split with `random_state=42` and stratification on $y$ to preserve the empirical class distribution across both partitions.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Partition Shapes:")
print(f"  • X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"  • X_test shape : {X_test.shape}  | y_test shape : {y_test.shape}")
print("-" * 50)
print("Training Target Distribution:")
print(y_train.value_counts(normalize=True) * 100)
print("\nTest Target Distribution:")
print(y_test.value_counts(normalize=True) * 100)

Partition Shapes:
  • X_train shape: (800, 20) | y_train shape: (800,)
  • X_test shape : (200, 20)  | y_test shape : (200,)
--------------------------------------------------
Training Target Distribution:
default
0    70.0
1    30.0
Name: proportion, dtype: float64

Test Target Distribution:
default
0    70.0
1    30.0
Name: proportion, dtype: float64


---
## SECTION 5 — ColumnTransformer Pipeline Construction

We construct a scikit-learn `ColumnTransformer`:
- **Numerical Pipeline**: `StandardScaler()` zero-mean unit-variance scaling.
- **Categorical Pipeline**: `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` to generate dense dummy indicator variables.

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='drop'
)
print("ColumnTransformer constructed successfully:")
print(preprocessor)

ColumnTransformer constructed successfully:
ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['duration_in_months', 'credit_amount',
                                  'installment_rate_pct_disposable_income',
                                  'present_residence_since', 'age_years',
                                  'existing_credits_count',
                                  'people_liable_maintenance']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['status_existing_checking_account',
                                  'credit_history', 'purpose',
                                  'savings_account_bonds',
                                  'present_employment_since',
                                  'personal_status_sex',
                                  'other_debtors_guarantor

---
## SECTION 6 — Fit Preprocessor Strictly on Training Data ($X_{train}$)

To prevent information leakage, the `ColumnTransformer` is fitted exclusively on $X_{train}$.

In [6]:
preprocessor.fit(X_train)
print("Preprocessor fitted strictly on X_train (Zero Data Leakage).")

Preprocessor fitted strictly on X_train (Zero Data Leakage).


---
## SECTION 7 — Transform $X_{train}$ and $X_{test}$

We transform both partitions into dense numeric feature matrices.

In [7]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

processed_feature_names = preprocessor.get_feature_names_out()

print(f"Transformation Results:")
print(f"  • X_train_processed shape : {X_train_processed.shape}")
print(f"  • X_test_processed shape  : {X_test_processed.shape}")
print(f"  • Processed Feature Count : {len(processed_feature_names)}")
print(f"  • Matrix Data Type        : {X_train_processed.dtype}")

Transformation Results:
  • X_train_processed shape : (800, 61)
  • X_test_processed shape  : (200, 61)
  • Processed Feature Count : 61
  • Matrix Data Type        : float64


---
## SECTION 8 — Processed Feature Inventory

We inspect the dynamically generated one-hot encoded feature names.

In [8]:
processed_df = pd.DataFrame(X_train_processed, columns=processed_feature_names)
print(f"First 10 Processed Features out of {len(processed_feature_names)}:")
for idx, name in enumerate(processed_feature_names[:10], 1):
    print(f"  {idx:02d}. {name}")
print("  ...")
print(f"Last 5 Processed Features:")
for idx, name in enumerate(processed_feature_names[-5:], len(processed_feature_names)-4):
    print(f"  {idx:02d}. {name}")

First 10 Processed Features out of 61:
  01. num__duration_in_months
  02. num__credit_amount
  03. num__installment_rate_pct_disposable_income
  04. num__present_residence_since
  05. num__age_years
  06. num__existing_credits_count
  07. num__people_liable_maintenance
  08. cat__status_existing_checking_account_A11
  09. cat__status_existing_checking_account_A12
  10. cat__status_existing_checking_account_A13
  ...
Last 5 Processed Features:
  57. cat__job_A174
  58. cat__telephone_A191
  59. cat__telephone_A192
  60. cat__foreign_worker_A201
  61. cat__foreign_worker_A202


---
## SECTION 9 — Automated Preprocessing Validation Gates

We execute 10 programmatic validation checks to guarantee preprocessing integrity.

In [9]:
print("Running Phase 2 Automated Validation Gates...")

# Gate 1: Train shape consistency
assert X_train_processed.shape[0] == len(X_train), "Train row count mismatch!"
print("  ✓ Gate 1: Training row count consistency verified.")

# Gate 2: Test shape consistency
assert X_test_processed.shape[0] == len(X_test), "Test row count mismatch!"
print("  ✓ Gate 2: Test row count consistency verified.")

# Gate 3: Feature dimension consistency between train and test
assert X_train_processed.shape[1] == X_test_processed.shape[1], "Train and Test feature dimension mismatch!"
print("  ✓ Gate 3: Feature dimension consistency verified.")

# Gate 4: Feature count matches get_feature_names_out()
assert X_train_processed.shape[1] == len(processed_feature_names), "Feature names count mismatch!"
print("  ✓ Gate 4: Feature count matches preprocessor.get_feature_names_out().")

# Gate 5: Zero NaNs in X_train_processed
assert not np.isnan(X_train_processed).any(), "NaN values found in X_train_processed!"
print("  ✓ Gate 5: Zero NaNs in X_train_processed confirmed.")

# Gate 6: Zero NaNs in X_test_processed
assert not np.isnan(X_test_processed).any(), "NaN values found in X_test_processed!"
print("  ✓ Gate 6: Zero NaNs in X_test_processed confirmed.")

# Gate 7: Zero infinite values
assert np.isfinite(X_train_processed).all() and np.isfinite(X_test_processed).all(), "Infinite values detected!"
print("  ✓ Gate 7: Zero infinite values confirmed.")

# Gate 8: Numerical matrix type
assert np.issubdtype(X_train_processed.dtype, np.floating), "Processed matrix is not float dtype!"
print("  ✓ Gate 8: Numerical floating-point matrix verified.")

# Gate 9: Target vector separation
assert y_train.shape[0] == X_train_processed.shape[0], "Target and feature vector length mismatch!"
print("  ✓ Gate 9: Target vector separation verified.")

# Gate 10: Stratified class balance preserved
train_def_rate = y_train.mean()
test_def_rate = y_test.mean()
assert abs(train_def_rate - test_def_rate) < 0.01, "Stratified split class proportion mismatch!"
print(f"  ✓ Gate 10: Stratified balance verified (Train: {train_def_rate*100:.1f}%, Test: {test_def_rate*100:.1f}%).")

print("\n" + "="*60)
print("ALL 10 PHASE 2 PREPROCESSING VALIDATION GATES PASSED!")
print("="*60)

Running Phase 2 Automated Validation Gates...
  ✓ Gate 1: Training row count consistency verified.
  ✓ Gate 2: Test row count consistency verified.
  ✓ Gate 3: Feature dimension consistency verified.
  ✓ Gate 4: Feature count matches preprocessor.get_feature_names_out().
  ✓ Gate 5: Zero NaNs in X_train_processed confirmed.
  ✓ Gate 6: Zero NaNs in X_test_processed confirmed.
  ✓ Gate 7: Zero infinite values confirmed.
  ✓ Gate 8: Numerical floating-point matrix verified.
  ✓ Gate 9: Target vector separation verified.
  ✓ Gate 10: Stratified balance verified (Train: 30.0%, Test: 30.0%).

ALL 10 PHASE 2 PREPROCESSING VALIDATION GATES PASSED!
